# 07 — Open set, label missingness, and active review

This notebook evaluates improved Stage-A models for `modeled ecotype` versus biological `Other`, estimates source-dependent label availability, runs an inverse-propensity sensitivity analysis, and creates a diverse internal review sample of unknown encounters.

`P_OTHER` is not abstention. It is the probability of a real non-SRKW/non-Transient ecotype. Because only 62 known-Other encounters and no blinded unknown-label audit sample are available, all unknown rows remain count-ineligible and retain unit expected unknown mass.

In [ ]:
import os
from pathlib import Path
import json
import pandas as pd

from experiment_support import resolve_release_paths
from open_set_missingness_experiment import run_open_set_missingness_experiment

paths = resolve_release_paths()
output_dir = Path(os.environ['MARINE_MAMMALS_RESEARCH_OUTPUT_ROOT']).expanduser().resolve() / 'open_set_missingness'
paths.release_id, paths.snapshot_id

In [ ]:
result = run_open_set_missingness_experiment(output_dir, paths)
manifest = json.loads(result['manifest_path'].read_text())
manifest['best_diagnostic_open_model'], manifest['promotion_eligible']

## Stage-A open-set candidates

The recall threshold is selected cross-fit to target at least 90% known-Other recall. Modeled retention measures the cost of that safety target.

In [ ]:
display(result['open_metrics'].sort_values(['log_loss', 'brier']).reset_index(drop=True))
display(result['multiclass'])

## Source-dependent label availability

The propensity model is a diagnostic of the observed missing-label mechanism. Inverse-propensity results assume labels are missing at random conditional on the measured fields; the data cannot verify that assumption.

In [ ]:
display(pd.DataFrame([result['propensity_metrics']]))
display(result['propensity_by_source'].sort_values('observed_label_rate').reset_index(drop=True))
display(result['binary_sensitivity'].sort_values(['evaluation_weight', 'log_loss']).reset_index(drop=True))

## Proposed blinded audit sample

These rows are acquisition priorities only. They are explicitly unreviewed, contribute no soft counts, and are balanced across source/era strata before filling by uncertainty priority.

In [ ]:
audit = result['active_sample']
display(audit.head(25))
display(audit.groupby(['SOURCE', 'ERA']).size().rename('n').reset_index())
assert not audit['SOFT_COUNT_ELIGIBLE'].any()
assert audit['EXPECTED_UNKNOWN_COUNT'].eq(1.0).all()

In [ ]:
print('Promotion blockers:')
for blocker in manifest['promotion_blockers']:
    print('-', blocker)